### Tugas 9 – Mini Klasifikasi Berdasarkan Dataset Kelompok
### Sistem Rekomendasi Film Berbasis AI
##### Kelompok 13 : Muhammad Rayhan Mumtaz | 2024081040 & Nia Koifa | 2024011001


---

## 1. Import Library

In [33]:
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.naive_bayes import MultinomialNB 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report

## 2. Load Dataset

In [34]:
# Baca dataset
data = pd.read_csv("Survei Preferensi Menonton_ Pengembangan Sistem Rekomendasi Film Berbasis AI .csv")

# Ambil minimal 20 data sesuai instruksi tugas
data = data.head(20).copy()

print(f"Jumlah data: {len(data)} baris")
print(f"Jumlah kolom: {len(data.columns)} kolom")
data.head(20)

Jumlah data: 20 baris
Jumlah kolom: 14 kolom


,Cap waktu,Nama Lengkap,Jenis Kelamin,Asal Universitas,Angkatan,Frekuensi Menonton Film dalam Seminggu:,Perangkat yang Digunakan,Bahasa Film yang Paling Disukai:,Genre Favorit (Pilih 3 Teratas):,Faktor Terpenting dalam Memilih Film (Skala 1-5): [Rating/Popularitas],Faktor Terpenting dalam Memilih Film (Skala 1-5): [Aktor/Aktris Terkenal],Faktor Terpenting dalam Memilih Film (Skala 1-5): [Alur Cerita (Plot Twist)],Faktor Terpenting dalam Memilih Film (Skala 1-5): [Kualitas Visual/CGI],"Skenario Minat: ""Jika ada film baru dengan genre favorit Anda namun memiliki rating rendah dari kritikus, apakah Anda tetap tertarik menontonnya?"""
0,2026/04/03 11:10:10 PM GMT+7,Rayhan Christian Wewengkang,Laki-laki,UPJ,2024,0 - 1 Kali,Laptop/PC,Asia - Korea/Jepang/Mandarin,Action;Horror;Romance;Animation,2,4,5,5,Ya (Tertarik)
1,2026/04/03 11:13:15 PM GMT+7,Ruud Zaki Bramdani,Laki-laki,Upj,2024,0 - 1 Kali,Laptop/PC,Indonesia,Horror;Comedy,2,1,3,3,Ya (Tertarik)
2,2026/04/03 11:28:32 PM GMT+7,Fizar Erlansyah,Laki-laki,Universitas Pembangunan Jaya,2024,0 - 1 Kali,Smartphone,Barat/Inggris,Action;Sci-Fi;Thriller,3,5,5,5,Ya (Tertarik)
3,2026/04/03 11:45:28 PM GMT+7,Laurensius Jovito Mahaputra Darsono,Laki-laki,Universitas Pembangunan Jaya,2024,0 - 1 Kali,Laptop/PC,Barat/Inggris,Action;Sci-Fi;Animation,1,2,3,3,Ya (Tertarik)
4,2026/04/04 12:34:16 PM GMT+7,Freyadiva,Perempuan,Universitas Pembangunan Jaya,2023,2 - 3 Kali,Laptop/PC,Asia - Korea/Jepang/Mandarin,Horror;Romance;Comedy,3,4,3,5,Ya (Tertarik)
5,2026/04/04 12:47:20 PM GMT+7,Dea,Perempuan,UPJ,2023,0 - 1 Kali,Laptop/PC,Asia - Korea/Jepang/Mandarin,Animation,4,1,5,4,Tidak (Tidak Tertarik)
6,2026/04/04 12:47:35 PM GMT+7,Adam Jagwani Syarif,Laki-laki,UPJ,2024,2 - 3 Kali,Laptop/PC,Barat/Inggris,Horror;Romance;Thriller,4,3,5,4,Tidak (Tidak Tertarik)
7,2026/04/04 12:50:05 PM GMT+7,Rexzy Febriano Chasan,Laki-laki,Universitas Pembangunan Jaya,2023,Setiap Hari,Smartphone,Asia - Korea/Jepang/Mandarin,Action;Comedy;Sci-Fi,3,1,4,5,Ya (Tertarik)
8,2026/04/04 1:11:58 PM GMT+7,Fitria Azzahra Nur Syahdiyah,Perempuan,Universitas Pembangunan Jaya Bintaro,2024,0 - 1 Kali,Smartphone,Indonesia,Action;Romance;Thriller,3,2,3,3,Tidak (Tidak Tertarik)
9,2026/04/04 1:30:13 PM GMT+7,Chelsea Meichika,Perempuan,Universitas Pembangunan Jaya,2024,0 - 1 Kali,Smartphone,Asia - Korea/Jepang/Mandarin,Action;Romance;Comedy,4,4,5,5,Tidak (Tidak Tertarik)


## 3. Pilih Fitur dan Label

In [35]:
# Nama kolom fitur
col_genre   = 'Genre Favorit (Pilih 3 Teratas): '
col_bahasa  = '  Bahasa Film yang Paling Disukai:  '
col_frek    = '  Frekuensi Menonton Film dalam Seminggu:  '
col_label   = 'Skenario Minat: "Jika ada film baru dengan genre favorit Anda namun memiliki rating rendah dari kritikus, apakah Anda tetap tertarik menontonnya?"'

# Gabungkan 3 fitur menjadi satu kolom teks
X = (
    data[col_genre].astype(str) + ' ' +
    data[col_bahasa].astype(str) + ' ' +
    data[col_frek].astype(str)
)

y = data[col_label]

print("Contoh data fitur (X):")
print(X.head(3).to_string())
print("\nLabel (y):")
print(y.head(3).to_string())

Contoh data fitur (X):
0    Action;Horror;Romance;Animation Asia - Korea/J...
1                   Horror;Comedy Indonesia 0 - 1 Kali
2      Action;Sci-Fi;Thriller Barat/Inggris 0 - 1 Kali

Label (y):
0    Ya (Tertarik)
1    Ya (Tertarik)
2    Ya (Tertarik)


## 4. Preprocessing

In [36]:
# Preprocessing: lowercase + hapus tanda baca
print("=== SEBELUM PREPROCESSING ===")
print(X.iloc[0])

X = X.str.lower()                          # Langkah 1: lowercase
X = X.str.replace(r'[;\-/]', ' ', regex=True)  # Langkah 2: hapus tanda baca
X = X.str.replace(r'\s+', ' ', regex=True).str.strip()  # Rapikan spasi berlebih

print("\n=== SESUDAH PREPROCESSING ===")
print(X.iloc[0])

=== SEBELUM PREPROCESSING ===
Action;Horror;Romance;Animation Asia - Korea/Jepang/Mandarin 0 - 1 Kali

=== SESUDAH PREPROCESSING ===
action horror romance animation asia korea jepang mandarin 0 1 kali


## 5. Vectorization – Ubah Teks ke Angka

In [37]:
# Ubah teks ke representasi angka menggunakan CountVectorizer
vectorizer = CountVectorizer()
X_vector = vectorizer.fit_transform(X)

print(f"Jumlah fitur (vocabulary): {len(vectorizer.vocabulary_)} kata unik")
print(f"Shape matrix fitur: {X_vector.shape}")
print("\nVocabulary:")
print(sorted(vectorizer.vocabulary_.keys()))

Jumlah fitur (vocabulary): 18 kata unik
Shape matrix fitur: (20, 18)

Vocabulary:
['action', 'animation', 'asia', 'barat', 'comedy', 'fi', 'hari', 'horror', 'indonesia', 'inggris', 'jepang', 'kali', 'korea', 'mandarin', 'romance', 'sci', 'setiap', 'thriller']


## 6. Split Data – Train & Test

In [38]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_vector, y, test_size=0.2, random_state=0
)

print(f"Jumlah data train : {X_train.shape[0]}")
print(f"Jumlah data test  : {X_test.shape[0]}")

Jumlah data train : 16
Jumlah data test  : 4


## 7. Training Model – Naive Bayes

In [39]:
# Buat dan latih model Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

print("Model berhasil dilatih!")
print(f"Kelas yang dikenali model: {model.classes_}")

Model berhasil dilatih!
Kelas yang dikenali model: ['Tidak (Tidak Tertarik)' 'Ya (Tertarik)']


## 8. Prediksi

In [40]:
# Lakukan prediksi pada data test
hasil = model.predict(X_test)

print("Hasil prediksi:")
print(hasil)

print("\nLabel asli (y_test):")
print(y_test.values)

print("\n--- Perbandingan Prediksi vs Aktual ---")
hasil_df = pd.DataFrame({
    'Aktual' : y_test.values,
    'Prediksi': hasil
})
hasil_df['Benar?'] = hasil_df['Aktual'] == hasil_df['Prediksi']
print(hasil_df.to_string(index=False))

# Evaluasi model
akurasi = accuracy_score(y_test, hasil)
print(f"Akurasi Model: {akurasi * 100:.1f}%")

print("\nClassification Report:")
print(classification_report(y_test, hasil))

Hasil prediksi:
['Tidak (Tidak Tertarik)' 'Ya (Tertarik)' 'Ya (Tertarik)' 'Ya (Tertarik)']

Label asli (y_test):
['Tidak (Tidak Tertarik)' 'Ya (Tertarik)' 'Ya (Tertarik)'
 'Tidak (Tidak Tertarik)']

--- Perbandingan Prediksi vs Aktual ---
                Aktual               Prediksi  Benar?
Tidak (Tidak Tertarik) Tidak (Tidak Tertarik)    True
         Ya (Tertarik)          Ya (Tertarik)    True
         Ya (Tertarik)          Ya (Tertarik)    True
Tidak (Tidak Tertarik)          Ya (Tertarik)   False
Akurasi Model: 75.0%

Classification Report:
                        precision    recall  f1-score   support

Tidak (Tidak Tertarik)       1.00      0.50      0.67         2
         Ya (Tertarik)       0.67      1.00      0.80         2

              accuracy                           0.75         4
             macro avg       0.83      0.75      0.73         4
          weighted avg       0.83      0.75      0.73         4

